[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nammaschool/interactive-learning/blob/main/notebooks/courses/integration_course.ipynb)

# Integration full course: practice notebook

Practice questions for every teaching chapter of the video, in the same order, and a graph you can move. Work each
question out on paper first, then type your answer into the box and press **check**. Press **show working** if
you'd like to see the steps.

**How the check works.** For an indefinite integral, the notebook differentiates your answer and compares it with
the integrand. So any correct form passes, whatever constant you chose, and you can leave out the + C. If it's
wrong, you'll see what your answer differentiates to, so you can spot the term that went astray.

**How to use this notebook.** The grey boxes are cells. Click the first one and press **Shift + Enter** (or the ▶ Run
button) to run it. Run each cell in turn, from the top down. When a slider appears, drag it and watch. Change any
number and run the cell again: nothing can break. It works in your browser (no install) and in Google Colab.

**Typing answers.** Use `*` or a space for times and `^` for powers: `x^4/4 + 2ln|x|`, `e^(3x)`, `sin(x)^2`,
`sqrt(x)`, `atan(x/3)`. Fractions as `14/3`, π as `pi`. For several answers, separate them with commas.

In [ ]:
# Setup: in the browser version (JupyterLite) the sliders need one install; elsewhere this does nothing.
import sys
if "pyodide" in sys.modules:
    import piplite
    await piplite.install(["ipywidgets"])

In [ ]:
# The look and the checker: run this cell first.
import math, cmath, re
import numpy as np
import matplotlib.pyplot as plt
import sympy as sp
from sympy.parsing.sympy_parser import parse_expr, standard_transformations, implicit_multiplication_application, convert_xor
from IPython.display import display
from ipywidgets import interact, IntSlider, Text, Button, Output, HBox, VBox, Layout

SLATE, SLATE2 = "#1f2b28", "#2a3a36"        # the slate ground
CHALK, DIM = "#ece6d6", "#93a097"           # chalk, and chalk smudge
CORAL = "#e3896b"                           # the one key thing in each figure
TEAL, SAGE, LILAC = "#6fb6bb", "#a0c27c", "#c9a0d3"   # the function · slopes · areas

plt.rcParams.update({
    "figure.facecolor": SLATE, "axes.facecolor": SLATE, "savefig.facecolor": SLATE,
    "text.color": CHALK, "axes.labelcolor": CHALK, "axes.edgecolor": DIM, "axes.titlecolor": CHALK,
    "xtick.color": DIM, "ytick.color": DIM, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": "#3b4c47", "grid.linewidth": 0.8,
    "font.size": 13, "axes.titlesize": 15, "axes.labelsize": 13, "legend.fontsize": 12,
    "legend.facecolor": SLATE2, "legend.edgecolor": DIM, "legend.labelcolor": CHALK, "legend.framealpha": 0.9,
    "lines.linewidth": 2.4, "figure.dpi": 100,
})

RULES = standard_transformations + (implicit_multiplication_application, convert_xor)
x = sp.Symbol("x")
r = sp.Rational
C_ = sp.Symbol("C")
NAMES = {"pi": sp.pi, "e": sp.E, "x": x, "C": C_, "c": C_, "ln": sp.log, "sqrt": sp.sqrt, "asin": sp.asin,
         "acos": sp.acos, "atan": sp.atan, "arcsin": sp.asin, "arctan": sp.atan, "sec": sp.sec, "cosec": sp.csc,
         "csc": sp.csc, "cot": sp.cot}

def read(s):
    """Your typed answer as exact expressions (a list, one for each answer)."""
    s = (s.replace("−", "-").replace("×", "*").replace("÷", "/").replace("π", "pi").replace("²", "^2")
          .replace("³", "^3"))
    for fn in ("sin", "cos", "tan"):                                  # tan⁻¹x and tan^-1 x → atan x
        s = s.replace(fn + "⁻¹", "a" + fn).replace(fn + "^-1", "a" + fn).replace(fn + "^(-1)", "a" + fn)
    s = re.sub(r"\|([^|]+)\|", r"(Abs(\1))", s)                       # ln|x| → ln(Abs(x))
    s = re.sub(r"√\s*(\d+(?:\.\d+)?|pi|\w)", r"sqrt(\1)", s).replace("√", "sqrt")   # √3 → sqrt(3); √(…) stays
    return [parse_expr(p, local_dict=NAMES, transformations=RULES) for p in s.split(",") if p.strip()]

POINTS = [0.3, 0.7, 1.3, 1.9, 2.6, 3.4, -0.4, -1.7, -2.6]   # where two expressions in x are compared

def same(a, b):
    """Equal as numbers, or (for expressions in x) equal at every sample point where the right answer is a real number."""
    a, b = sp.sympify(a), sp.sympify(b)
    if (a.free_symbols | b.free_symbols) - {x}: return False         # a letter we don't know (or a C left in)
    tried = 0
    for v in POINTS:
        try: wb = complex(sp.N(b.subs(x, v), 30))
        except (TypeError, ValueError): continue
        if not cmath.isfinite(wb) or abs(wb.imag) > 1e-12: continue    # the right answer isn't defined here
        try: wa = complex(sp.N(a.subs(x, v), 30))
        except (TypeError, ValueError): return False
        if not abs(wa - wb) < 1e-9 * max(1, abs(wb)): return False
        tried += 1
    return tried >= 3

def nice(e):
    """An expression the way you'd type it."""
    return (sp.sstr(e).replace("**", "^").replace("exp(", "e^(").replace("log(", "ln(").replace("Abs(", "abs(")
            .replace("*", "·"))

def readable(e):
    """The tidiest of a few forms: term by term if that isn't much longer."""
    forms = [nice(sp.expand(e)), nice(e), nice(sp.trigsimp(e))]
    short = min(len(s) for s in forms)
    return next(s for s in forms if len(s) <= 1.3 * short)

def antiderivative(F, f):
    """F is an antiderivative of f: its derivative equals f wherever f is defined."""
    return same(sp.diff(F.replace(sp.Abs, lambda a: a), x), f)

# Each question: the question and either the integrand (an indefinite integral: your answer is differentiated and
# compared with it) or the exact answers in order; and the working.
Q = {}

def practice(n, answer=""):
    """Question n: an answer box, a check button and a show-working button."""
    q = Q[n]
    box = Text(value=answer, placeholder="your answer", description="answer:", layout=Layout(width="460px"))
    check, work, out = Button(description="check"), Button(description="show working"), Output()
    def on_check(_=None):
        out.clear_output()
        with out:
            try: got = read(box.value)
            except Exception: print("I can't read that yet: check the brackets."); return
            if "integrand" in q:
                if len(got) != 1: print("✗ type one answer: the integral, with or without + C."); return
                d = sp.diff(got[0].replace(sp.Abs, lambda a: a), x)        # ln|u| and ln(u): the same derivative
                f = q["integrand"]
                print("Your answer differentiates to:  ", readable(d))
                print("The integrand is:               ", nice(f))
                if same(d, f): print("✓ right: they match, so your answer is an integral (any constant passes).")
                else: print("✗ not quite: they differ. Compare them term by term, or press show working.")
                return
            want = q["answer"]
            if len(got) != len(want): print(f"✗ the question asks for {len(want)} answer(s); you typed {len(got)}."); return
            ok = all(same(g, w) for g, w in zip(got, want))
            print("✓ right" if ok else "✗ not quite: try again, or press show working")
    def on_work(_=None):
        out.clear_output()
        with out:
            for line in q["working"]: print(line)
    check.on_click(on_check); work.on_click(on_work)
    print(f"Question {n}. {q['text']}")
    display(VBox([HBox([box, check, work]), out]))
    if answer: on_check()

## 1 · Integration as the reverse of differentiation

An antiderivative of f is a function F with F′ = f. They come as a family, F(x) + C, and one starting value picks
out a single member of the family.

In [ ]:
Q[1] = {"text": "F′(x) = 6x² − 2, and F(1) = 3. Find F(x). (This one has a starting value, so the constant counts.)",
        "answer": [2 * x ** 3 - 2 * x + 3],
        "working": ["Reverse the power rule: F(x) = 2x³ − 2x + C. (Check: its derivative is 6x² − 2.)",
                    "F(1) = 2 − 2 + C = C, and that must be 3.",
                    "So F(x) = 2x³ − 2x + 3."]}
assert antiderivative(2 * x ** 3 - 2 * x + 3, 6 * x ** 2 - 2) and (2 * x ** 3 - 2 * x + 3).subs(x, 1) == 3
practice(1)

## 2 · Standard integrals

Every row of the table is a derivative read backwards: ∫xⁿ dx = xⁿ⁺¹/(n + 1) (n ≠ −1), ∫1/x dx = ln|x|,
∫eˣ dx = eˣ, ∫cos x dx = sin x, ∫sin x dx = −cos x, ∫sec²x dx = tan x. Integrals split over sums, and constants
come outside. Three integrals from the table, then one that needs tidying first.

In [ ]:
Q["2a"] = {"text": "∫ (x³ + 4/x) dx", "integrand": x ** 3 + 4 / x,
           "working": ["Term by term: ∫x³ dx = x⁴/4, and ∫4/x dx = 4 ln|x|.", "x⁴/4 + 4 ln|x| + C."]}
assert antiderivative(x ** 4 / 4 + 4 * sp.log(sp.Abs(x)), Q["2a"]["integrand"])
practice("2a")

In [ ]:
Q["2b"] = {"text": "∫ (3 cos x − 2eˣ) dx", "integrand": 3 * sp.cos(x) - 2 * sp.exp(x),
           "working": ["∫3 cos x dx = 3 sin x, and ∫2eˣ dx = 2eˣ.", "3 sin x − 2eˣ + C."]}
assert antiderivative(3 * sp.sin(x) - 2 * sp.exp(x), Q["2b"]["integrand"])
practice("2b")

In [ ]:
Q["2c"] = {"text": "∫ (sec²x + 6√x) dx", "integrand": sp.sec(x) ** 2 + 6 * sp.sqrt(x),
           "working": ["∫sec²x dx = tan x.", "6√x = 6x^(1/2), so its integral is 6 × x^(3/2)/(3/2) = 4x^(3/2).",
                       "tan x + 4x^(3/2) + C."]}
assert antiderivative(sp.tan(x) + 4 * x ** r(3, 2), Q["2c"]["integrand"])
practice("2c")

In [ ]:
Q["2d"] = {"text": "∫ (√x + 1)²/x dx   (tidy it up first)", "integrand": (sp.sqrt(x) + 1) ** 2 / x,
           "working": ["Expand the top: x + 2√x + 1.", "Divide each term by x: 1 + 2/√x + 1/x = 1 + 2x^(−1/2) + 1/x.",
                       "Now each term is in the table: x + 4√x + ln|x| + C."]}
assert antiderivative(x + 4 * sp.sqrt(x) + sp.log(sp.Abs(x)), Q["2d"]["integrand"])
practice("2d")

## 3 · Integration by substitution

Substitution is the chain rule backwards. Spot an inside function u whose derivative is also in the integrand, swap
to u, integrate, then swap back.

In [ ]:
Q[3] = {"text": "∫ x²(x³ + 1)⁴ dx", "integrand": x ** 2 * (x ** 3 + 1) ** 4,
        "working": ["Let u = x³ + 1. Then du = 3x² dx, so x² dx = du/3.", "∫u⁴ du/3 = u⁵/15.", "(x³ + 1)⁵/15 + C."]}
assert antiderivative((x ** 3 + 1) ** 5 / 15, Q[3]["integrand"])
practice(3)

## 4 · Integrals using trigonometric identities

Change what you can't integrate into what you can: sin²x = (1 − cos 2x)/2, cos²x = (1 + cos 2x)/2, and products
of sines and cosines become sums.

In [ ]:
Q[4] = {"text": "∫ sin²(3x) dx", "integrand": sp.sin(3 * x) ** 2,
        "working": ["sin²(3x) = (1 − cos 6x)/2.", "∫(1 − cos 6x)/2 dx = x/2 − sin(6x)/12.", "x/2 − sin(6x)/12 + C."]}
assert antiderivative(x / 2 - sp.sin(6 * x) / 12, Q[4]["integrand"])
practice(4)

## 5 · Integration by parts

∫u dv = uv − ∫v du. Choose u to get simpler when you differentiate it, and dv to be something you can integrate.
Choose your u and dv, type your answer, and check it. If it's wrong, the check shows what your answer
differentiates to.

In [ ]:
Q[5] = {"text": "∫ x e^(3x) dx", "integrand": x * sp.exp(3 * x),
        "working": ["u = x, so du = dx. dv = e^(3x) dx, so v = e^(3x)/3.",
                    "uv − ∫v du = x e^(3x)/3 − ∫e^(3x)/3 dx.",
                    "= x e^(3x)/3 − e^(3x)/9 + C."]}
assert antiderivative(x * sp.exp(3 * x) / 3 - sp.exp(3 * x) / 9, Q[5]["integrand"])
practice(5)

## 6 · Integration by partial fractions

Split a fraction whose bottom factorises into simple pieces, A/(x − a) + B/(x − b), then integrate each to a log.

In [ ]:
Q[6] = {"text": "∫ (4x + 1)/((x − 2)(x + 1)) dx", "integrand": (4 * x + 1) / ((x - 2) * (x + 1)),
        "working": ["(4x + 1)/((x − 2)(x + 1)) = A/(x − 2) + B/(x + 1), so 4x + 1 = A(x + 1) + B(x − 2).",
                    "x = 2: 9 = 3A, so A = 3. x = −1: −3 = −3B, so B = 1.",
                    "3 ln|x − 2| + ln|x + 1| + C."]}
assert sp.simplify(3 / (x - 2) + 1 / (x + 1) - Q[6]["integrand"]) == 0
assert antiderivative(3 * sp.log(sp.Abs(x - 2)) + sp.log(sp.Abs(x + 1)), Q[6]["integrand"])
practice(6)

## 7 · Integrals of some particular functions

∫1/(x² + a²) dx = (1/a) tan⁻¹(x/a). A quadratic on the bottom with no real roots: complete the square first.

In [ ]:
Q[7] = {"text": "∫ 1/(x² + 4x + 13) dx", "integrand": 1 / (x ** 2 + 4 * x + 13),
        "working": ["Complete the square: x² + 4x + 13 = (x + 2)² + 9 = (x + 2)² + 3².",
                    "With u = x + 2 and a = 3: (1/3) tan⁻¹((x + 2)/3) + C."]}
assert sp.expand((x + 2) ** 2 + 9) == x ** 2 + 4 * x + 13
assert antiderivative(sp.atan((x + 2) / 3) / 3, Q[7]["integrand"])
practice(7)

## 8 · The definite integral

Cut the area under y = x² from 0 to 1 into strips, and make each strip a rectangle. Drag the slider to make the
strips thinner: the totals with right-edge heights and with left-edge heights trap the area between them, and both
close in on 1/3. That limit is the definite integral.

In [ ]:
def right(n, a=0.0, b=1.0, f=lambda v: v * v): return sum(f(a + i * (b - a) / n) for i in range(1, n + 1)) * (b - a) / n
def left(n, a=0.0, b=1.0, f=lambda v: v * v): return sum(f(a + i * (b - a) / n) for i in range(0, n)) * (b - a) / n
assert abs(right(4) - 0.46875) < 1e-12 and abs(left(4) - 0.21875) < 1e-12                  # the video's four strips
assert round(right(10), 3) == 0.385 and round(left(10), 3) == 0.285                         # and ten
assert all(abs(right(n) - left(n) - 1 / n) < 1e-12 and left(n) < 1 / 3 < right(n) for n in (4, 10, 100))

def strips(n=4):
    xs = np.linspace(0, 1.05, 300)
    w = 1 / n
    fig, ax = plt.subplots(figsize=(8.6, 5))
    ax.bar([i * w for i in range(n)], [((i + 1) * w) ** 2 for i in range(n)], width=w, align="edge",
           color=LILAC, alpha=0.30, edgecolor=LILAC, lw=1 if n <= 40 else 0, label="right-edge strips")
    ax.bar([i * w for i in range(n)], [(i * w) ** 2 for i in range(n)], width=w, align="edge",
           color=LILAC, alpha=0.55, lw=0, label="left-edge strips")
    ax.plot(xs, xs * xs, color=TEAL, label="y = x²")
    ax.set_xlim(0, 1.05); ax.set_ylim(0, 1.1); ax.set_xlabel("x"); ax.legend(loc="upper left")
    ax.set_title(f"{n} strips:  {left(n):.5f}  <  area  <  {right(n):.5f}     (1/3 = 0.33333)")
    plt.show()

In [ ]:
interact(strips, n=IntSlider(value=4, min=1, max=100, step=1, description="strips"));

In [ ]:
Q[8] = {"text": "For y = x² from 0 to 2: give the right-edge total with 4 strips, then the exact area (the limit "
                "as the strips get thinner).",
        "answer": [r(15, 4), r(8, 3)],
        "working": ["Four strips of width 0.5. Right-edge heights: 0.25, 1, 2.25, 4.",
                    "Total: (0.25 + 1 + 2.25 + 4) × 0.5 = 7.5 × 0.5 = 3.75 = 15/4.",
                    "The exact area: ∫₀² x² dx = 2³/3 = 8/3, about 2.667. The right-edge total overshoots it."]}
assert abs(right(4, 0, 2) - 3.75) < 1e-12 and sp.integrate(x ** 2, (x, 0, 2)) == r(8, 3)
practice(8)

## 9 · The fundamental theorem of calculus

If F′ = f, then ∫ₐᵇ f(x) dx = F(b) − F(a). Find any antiderivative and subtract its values at the two ends.

In [ ]:
Q[9] = {"text": "Evaluate ∫₁⁴ √x dx, then ∫₀^(π/2) cos x dx. Give the two values, in that order.",
        "answer": [r(14, 3), 1],
        "working": ["An antiderivative of √x is (2/3)x^(3/2).",
                    "(2/3)(8) − (2/3)(1) = 16/3 − 2/3 = 14/3.",
                    "An antiderivative of cos x is sin x: sin(π/2) − sin 0 = 1 − 0 = 1."]}
assert sp.integrate(sp.sqrt(x), (x, 1, 4)) == r(14, 3) and sp.integrate(sp.cos(x), (x, 0, sp.pi / 2)) == 1
practice(9)

## 10 · Properties of definite integrals

On an interval balanced around 0, odd functions cancel (∫₋ₐᵃ odd = 0) and even ones double. And
∫₀ᵃ f(x) dx = ∫₀ᵃ f(a − x) dx.

In [ ]:
Q[10] = {"text": "Evaluate ∫₋₂² (x³ + 4x² − x) dx. Then use the a − x property to evaluate "
                 "∫₀^(π/2) sin x/(sin x + cos x) dx. Give the two values, in that order.",
         "answer": [r(64, 3), sp.pi / 4],
         "working": ["x³ and −x are odd: they cancel from −2 to 2. 4x² is even: 2 × ∫₀² 4x² dx = 2 × 32/3 = 64/3.",
                     "Call the second one I. Replacing x by π/2 − x swaps sin and cos: I = ∫₀^(π/2) cos x/(cos x + sin x) dx.",
                     "Add the two forms: 2I = ∫₀^(π/2) 1 dx = π/2, so I = π/4."]}
assert sp.integrate(x ** 3 + 4 * x ** 2 - x, (x, -2, 2)) == r(64, 3)
_g = lambda v: math.sin(v) / (math.sin(v) + math.cos(v))
assert abs(sum(_g((i + 0.5) * math.pi / 2 / 20000) for i in range(20000)) * math.pi / 2 / 20000 - math.pi / 4) < 1e-9
practice(10)

## 11 · Area under and between curves

The area between two curves is ∫ (top − bottom) dx, from where they cross to where they cross again.

In [ ]:
Q[11] = {"text": "Find the area between the line y = x and the curve y = x² − 2x.",
         "answer": [r(9, 2)],
         "working": ["They cross where x = x² − 2x, so x² − 3x = 0: x = 0 and x = 3.",
                     "Between them the line is on top: x − (x² − 2x) = 3x − x².",
                     "∫₀³ (3x − x²) dx = 27/2 − 9 = 9/2."]}
assert sp.solve(sp.Eq(x, x ** 2 - 2 * x), x) == [0, 3] and sp.integrate(3 * x - x ** 2, (x, 0, 3)) == r(9, 2)
practice(11)

## 12 · Volumes of revolution

Spin y = f(x) about the x-axis: each thin slice is a disc of radius y, so the volume is ∫ π y² dx.

In [ ]:
Q[12] = {"text": "The curve y = x², from x = 0 to x = 2, is spun about the x-axis. Find the volume, exactly.",
         "answer": [32 * sp.pi / 5],
         "working": ["V = ∫₀² π (x²)² dx = π ∫₀² x⁴ dx.", "= π × 2⁵/5 = 32π/5, about 20.1."]}
assert sp.integrate(sp.pi * x ** 4, (x, 0, 2)) == 32 * sp.pi / 5
practice(12)

## 13 · Differential equations

A separable equation, dy/dx = g(x) h(y): put every y on one side and every x on the other, integrate both sides,
then use the starting value to find the constant.

In [ ]:
Q[13] = {"text": "dy/dx = 2xy, and y = 3 when x = 0. Find y as a function of x.",
         "answer": [3 * sp.exp(x ** 2)],
         "working": ["Separate: dy/y = 2x dx.", "Integrate: ln|y| = x² + C, so y = A e^(x²).",
                     "At x = 0, y = A = 3. So y = 3e^(x²)."]}
_y = 3 * sp.exp(x ** 2)
assert sp.simplify(sp.diff(_y, x) - 2 * x * _y) == 0 and _y.subs(x, 0) == 3
practice(13)

## Sources
NCERT Mathematics Class XII, ch. 7 (Integrals), ch. 8 (Application of Integrals) and ch. 9 (Differential Equations);
OpenStax, Calculus Volume 1, ch. 5–6, and Volume 2, ch. 3. All retrieved 2026-10-31. Every question is our own.